# الخطوة 1: استيراد المكتبات وضبط إعدادات الرسوم البيانية

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# الخطوة 2: تحميل البيانات والفحص الهيكلي الأولي

In [ ]:
# قراءة الملفات
df_eg = pd.read_csv("eg_drugs.csv")
df_inter = pd.read_csv("db_drug_interactions.csv")

print("--- أبعاد البيانات ---")
print(f"eg_drugs: {df_eg.shape[0]} صف و {df_eg.shape[1]} عمود")
print(f"db_drug_interactions: {df_inter.shape[0]} صف و {df_inter.shape[1]} عمود\n")

print("--- معاينة eg_drugs ---")
print(df_eg.info())
display(df_eg.head(3))

print("\n--- معاينة db_drug_interactions ---")
print(df_inter.info())
display(df_inter.head(3))

# الخطوة 3: فحص جودة البيانات (Data Quality Assessment)

In [ ]:
# 3.1 القيم المفقودة (Missing Values)
print("--- القيم المفقودة في أدوية السوق المصري ---")
missing_eg = (
    df_eg.isnull().sum()[df_eg.isnull().sum() > 0].sort_values(ascending=False)
)
print(
    missing_eg
    if not missing_eg.empty
    else "لا توجد قيم مفقودة في الأعمدة الأساسية."
)

print("\n--- القيم المفقودة في قاعدة تداخلات الأدوية ---")
missing_inter = df_inter.isnull().sum()
print(missing_inter[missing_inter > 0])

# 3.2 فحص التكرارات (Duplicates)
print(f"\nالصفوف المكررة بالكامل في eg_drugs: {df_eg.duplicated().sum()}")
print(
    f"الصفوف المكررة بالكامل في db_drug_interactions: {df_inter.duplicated().sum()}"
)

# 3.3 تجهيز وتوحيد النصوص لتسهيل الربط (String Standardization)
# تنظيف المسافات الزائدة وتحويل النصوص إلى أحرف صغيرة
text_cols_eg = df_eg.select_dtypes(include="object").columns
for col in text_cols_eg:
    df_eg[col] = df_eg[col].astype(str).str.strip().str.lower()

text_cols_inter = df_inter.select_dtypes(include="object").columns
for col in text_cols_inter:
    df_inter[col] = df_inter[col].astype(str).str.strip().str.lower()

# الخطوة 4: التحليل الاستكشافي الأحادي (Univariate Analysis)

In [ ]:
# --- أولاً: بيانات أدوية السوق المصري (eg_drugs) ---
# تحديد عمود المادة الفعالة أو الشكل الدوائي بحسب التسمية الشائعة
active_ing_col = [
    col
    for col in df_eg.columns
    if any(k in col for k in ["active", "ingredient", "generic", "substance"])
]
active_col = active_ing_col[0] if active_ing_col else df_eg.columns[1]

# أعلى المواد الفعالة تكراراً
top_active = df_eg[active_col].value_counts().head(10)

plt.figure(figsize=(10, 4))
sns.barplot(x=top_active.values, y=top_active.index, palette="Blues_r")
plt.title(f"Top 10 Active Ingredients in eg_drugs ({active_col})")
plt.xlabel("Count")
plt.ylabel("Active Ingredient")
plt.tight_layout()
plt.show()

# فحص المتغيرات الرقمية (مثل السعر إن وجد)
num_cols = df_eg.select_dtypes(include=[np.number]).columns
if len(num_cols) > 0:
    print("\n--- الإحصاءات الوصفية للمتغيرات الرقمية ---")
    display(df_eg[num_cols].describe().T)

    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df_eg[num_cols[0]], color="skyblue")
    plt.title(f"Distribution of {num_cols[0]}")
    plt.tight_layout()
    plt.show()

# --- ثانياً: بيانات التفاعلات الدوائية (db_drug_interactions) ---
# تحديد أعمدة الأدوية المتفاعلة
drug_cols = [
    col
    for col in df_inter.columns
    if any(k in col for k in ["drug", "name", "substance"])
]
drug1_col = drug_cols[0] if len(drug_cols) > 0 else df_inter.columns[0]
drug2_col = drug_cols[1] if len(drug_cols) > 1 else df_inter.columns[1]

# أكثر الأدوية ظهوراً في شبكة التفاعلات
all_interacting_drugs = pd.concat([df_inter[drug1_col], df_inter[drug2_col]])
top_interacting = all_interacting_drugs.value_counts().head(10)

plt.figure(figsize=(10, 4))
sns.barplot(x=top_interacting.values, y=top_interacting.index, palette="Reds_r")
plt.title("Top 10 Drugs with Most Documented Interactions")
plt.xlabel("Total Interactions Recorded")
plt.ylabel("Drug Name")
plt.tight_layout()
plt.show()

# فحص عمود درجة الخطورة أو نوع التفاعل إذا كان متوفراً
severity_col = [
    col
    for col in df_inter.columns
    if any(k in col for k in ["severity", "level", "type", "risk"])
]
if severity_col:
    plt.figure(figsize=(7, 3))
    sns.countplot(
        data=df_inter,
        x=severity_col[0],
        palette="viridis",
        order=df_inter[severity_col[0]].value_counts().index,
    )
    plt.title(f"Distribution of Interaction {severity_col[0].capitalize()}")
    plt.xlabel("Risk Level")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# الخطوة 5: التحليل الثنائي ودراسة ربط الملفين (Cross-Dataset Integration)

In [ ]:
# استخراج الكيانات الفريدة من كلا المصدرين
unique_eg_ingredients = set(df_eg[active_col].unique())
unique_interaction_drugs = set(all_interacting_drugs.unique())

# حساب التداخل والتقاطع
overlap = unique_eg_ingredients.intersection(unique_interaction_drugs)
match_rate = (len(overlap) / len(unique_eg_ingredients)) * 100

print("--- دراسة جدوى ربط المجموعتين (Data Linking Feasibility) ---")
print(f"إجمالي المواد الفعالة الفريدة في مصر: {len(unique_eg_ingredients)}")
print(f"إجمالي الأدوية الفريدة في قاعدة التداخلات: {len(unique_interaction_drugs)}")
print(f"المواد المطابقة في القاعدتين: {len(overlap)}")
print(f"نسبة التغطية (Match Rate): {match_rate:.2f}%")

# تجربة دمج تجريبي (Inner Join) لمعرفة حجم البيانات المترابطة
merged_sample = pd.merge(
    df_eg,
    df_inter,
    left_on=active_col,
    right_on=drug1_col,
    how="inner",
)
print(f"\nعدد التداخلات الموثقة لأدوية السوق المصري: {merged_sample.shape[0]}")
display(merged_sample.head(3))